# MFE-FCGAT, dementia vs HC: AD + FTD vs healthy controls on ds004504

Pre-registered in `docs/PREREG_2026-11-16.md` (part C). This is `GAT_multiclass_ds004504.ipynb` with AD and FTD
pooled into one **dementia** label: 88 subjects, 2 outputs, class-weighted loss. It is stage 1 of the two-stage
3-class model.

- **The folds are the 3-class notebook's folds.** They are built from the 3-class labels, and Cell 4 checks them
  line by line against that notebook's printed folds. Its outputs can therefore be combined patient by patient with
  the AD-vs-FTD stage.
- **Identical code** for the cache, node features, graphs, model (apart from the 2-way output), training (lr 3e-3,
  30 fixed epochs) and the band-power floor and soft vote.
- **Reported:** subject accuracy (argmax), balanced accuracy, sensitivity (dementia found), specificity (HC found),
  and AUC, also within AD and within FTD. Benchmark: Stefanou et al. 2025, 80.7 % (quoted from the AHEPA review).
- The statistics are in `Final_scoreboard_ds004504.ipynb`.

Reads `subject_cache/`; writes `results_gat_dementia/`. About 25 min on CPU.

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS, PATHS, DEVICE
# ===========================================================================
import json, math, random, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

WORK_DIR   = Path.cwd()
CACHE_DIR  = WORK_DIR / "subject_cache"
RESULT_DIR = WORK_DIR / "results_gat_dementia"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f"torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"cache   : {CACHE_DIR}  ({len(list(CACHE_DIR.glob('sub-*.npz')))} files)")
print(f"results : {RESULT_DIR}")

torch 2.14.0+cpu | device cpu
cache   : C:\Projects\Major Project\subject_cache  (88 files)
results : C:\Projects\Major Project\results_gat_dementia


In [2]:
# ===========================================================================
# CELL 2 — CONFIGURATION  (the only cell you normally edit)
# ===========================================================================
SEED = 42

# --- data: fixed by the cache ----------------------------------------------
BAND_NAMES  = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
BAND_DIMS   = {"Delta": 18, "Theta": 21, "Alpha": 26, "Beta": 86, "Gamma": 76}
N_NODES     = 19
N_SUBJECTS  = 88
CLASS_NAMES = ["HC", "dementia"]              # training label 0, 1 (AD and FTD pooled)
HC, AD, FTD = 0, 1, 2                         # the cache's 3-class codes
N_CLASSES   = len(CLASS_NAMES)
CLASSES     = list(range(N_CLASSES))
LABEL_OF    = lambda i: AD if i <= 36 else HC if i <= 65 else FTD   # sub-number -> label

# --- model: identical to GAT_ds004504 except the 3-way output ---------------
CNN_OUT_DIM  = 16
GAT_HIDDEN   = 16
GAT_OUT      = 5
HEADS        = 4
ATTN_DROPOUT = 0.1
MLP_HIDDEN   = (30, 14)
MLP_DROPOUT  = (0.8, 0.5)

# --- training -----------------------------------------------------------------
LR           = 3e-3        # the binary GAT's rate; not yet re-checked for 3 classes
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 64
EPOCHS       = 30          # fixed budget and cosine horizon: no validation set, no early stop.
                           # Chosen from the first run's VALIDATION curves (best epochs
                           # 38/15/20/28/20, flat after ~20), never from test results.
GRAD_CLIP    = 5.0
CLASS_WEIGHTED_LOSS = True # weight = n / (3 * n_class) on the training windows

# --- protocol -----------------------------------------------------------------
N_FOLDS   = 5
N_REPEATS = 3              # pooled numbers are quoted as mean +- SD over repeats

assert [BAND_DIMS[b] - CNN_OUT_DIM + 1 for b in BAND_NAMES] == [3, 6, 11, 71, 61]
seed_all(SEED)
print(f"MFE-FCGAT dementia vs HC | lr {LR} | {EPOCHS} fixed epochs | {N_REPEATS} x {N_FOLDS}-fold, "
      f"subject-level | class-weighted loss {CLASS_WEIGHTED_LOSS}")

MFE-FCGAT dementia vs HC | lr 0.003 | 30 fixed epochs | 3 x 5-fold, subject-level | class-weighted loss True


In [3]:
# ===========================================================================
# CELL 3 — LOAD THE FEATURE CACHE  (all 88 subjects)
# ===========================================================================
x_parts = {b: [] for b in BAND_NAMES}
e1_parts, e2_parts, bp_parts, y_parts, sid_parts = [], [], [], [], []

for i in range(1, N_SUBJECTS + 1):
    sid = f"sub-{i:03d}"
    with np.load(CACHE_DIR / f"{sid}.npz") as z:
        for b in BAND_NAMES:
            assert z[f"x_{b}"].shape[1:] == (N_NODES, BAND_DIMS[b]), (sid, b)
            x_parts[b].append(z[f"x_{b}"])
        e1_parts.append(z["e1"]); e2_parts.append(z["e2"])
        bp_parts.append(z["bp"]); y_parts.append(z["y"])
        sid_parts.append(np.full(len(z["y"]), sid))
    assert int(y_parts[-1][0]) == LABEL_OF(i), f"{sid}: wrong label"

X   = [np.concatenate(x_parts[b]).astype(np.float32) for b in BAND_NAMES]
E1  = np.concatenate(e1_parts).astype(np.float32)
E2  = np.concatenate(e2_parts).astype(np.float32)
BP  = np.concatenate(bp_parts).astype(np.float32)
Y3  = np.concatenate(y_parts).astype(np.int64)   # HC 0, AD 1, FTD 2: the folds are built from these
Y   = (Y3 != HC).astype(np.int64)                # the training target: 0 HC, 1 dementia
SID = np.concatenate(sid_parts)
del x_parts, e1_parts, e2_parts, bp_parts, y_parts, sid_parts

Xt  = [torch.from_numpy(a).to(DEVICE) for a in X]
E1t = torch.from_numpy(E1).to(DEVICE)
E2t = torch.from_numpy(E2).to(DEVICE)
Yt  = torch.from_numpy(Y).to(DEVICE)

_subs = np.unique(SID)
_lab  = np.array([Y[SID == s][0] for s in _subs])
print(f"windows  : {len(Y):,}  " + "  ".join(f"{n} {int((Y == c).sum()):,}" for c, n in enumerate(CLASS_NAMES)))
print(f"subjects : {len(_subs)}  " + "  ".join(f"{n} {int((_lab == c).sum())}" for c, n in enumerate(CLASS_NAMES)))


windows  : 8,050  HC 2,900  dementia 5,150
subjects : 88  HC 29  dementia 59


In [4]:
# ===========================================================================
# CELL 4 — SPLITS AND METRICS
# ===========================================================================
def make_splits(n_folds=N_FOLDS, n_repeats=N_REPEATS, seed=SEED):
    """Grouped, stratified K-fold over subjects; each fold trains on every non-test
    subject. Repeat r uses seed + 1000 r, so repeat 1 has the same test folds as
    the first (validation-selected) run."""
    splits, idx = [], np.arange(len(Y))
    for r in range(1, n_repeats + 1):
        outer = StratifiedGroupKFold(n_folds, shuffle=True, random_state=seed + 1000 * r)
        for f, (train, test) in enumerate(outer.split(idx, Y3, groups=SID), start=1):
            assert not set(SID[train]) & set(SID[test]), "SUBJECT LEAKAGE"
            splits.append(dict(repeat=r, fold=f, train=train, test=test))
    return splits


def _metrics(y3, P):
    """y3 (n,) 3-class labels, P (n, 2) = [P(HC), P(dementia)]. Accuracy at argmax, AUC of P(dementia)."""
    y, pred, sc = (y3 != HC).astype(int), P.argmax(1), P[:, 1]
    sens, spec = (pred[y == 1] == 1).mean(), (pred[y == 0] == 0).mean()
    def pair(g):
        m = np.isin(y3, [g, HC])
        return roc_auc_score(y3[m] == g, sc[m]) if len(np.unique(y3[m])) == 2 else np.nan
    return dict(acc=accuracy_score(y, pred), bal_acc=(sens + spec) / 2, sensitivity=sens, specificity=spec,
                auc=roc_auc_score(y, sc) if 0 < y.sum() < len(y) else np.nan,
                auc_AD_HC=pair(AD), auc_FTD_HC=pair(FTD))


def subject_probs(idx, P):
    """Mean class probabilities per subject. Returns subjects, 3-class labels, (n_subj, 2)."""
    sid, y = SID[idx], Y3[idx]
    subs = np.unique(sid)
    return subs, np.array([y[sid == s][0] for s in subs]), np.stack([P[sid == s].mean(0) for s in subs])


def score(idx, P):
    """Window-level (w_*) and subject-level (s_*) metrics."""
    _, s_y, s_P = subject_probs(idx, P)
    return {**{f"w_{k}": v for k, v in _metrics(Y3[idx], P).items()},
            **{f"s_{k}": v for k, v in _metrics(s_y, s_P).items()}}


def mean_sd(runs):
    """Mean and SD over repeats of a DataFrame with one row of pooled scores per repeat."""
    return runs.mean(), (runs.std(ddof=1) if len(runs) > 1 else runs.mean() * 0)


SPLITS = make_splits()
_ns  = lambda i: len(np.unique(SID[i]))
_cls = lambda i: "/".join(str(len(np.unique(SID[i][Y3[i] == c]))) for c in (HC, AD, FTD))
for s in SPLITS:
    print(f"r{s['repeat']} f{s['fold']}: train {len(s['train']):5d}w/{_ns(s['train']):2d}s ({_cls(s['train'])}) | "
          f"test {len(s['test']):4d}w/{_ns(s['test']):2d}s ({_cls(s['test'])})   [HC/AD/FTD]")
for r in range(1, N_REPEATS + 1):
    _tested = np.concatenate([s["test"] for s in SPLITS if s["repeat"] == r])
    assert len(_tested) == len(np.unique(_tested)) == len(Y)
# the 3-class notebook printed its folds; these must match them line for line
_ref = json.loads((WORK_DIR / "GAT_multiclass_ds004504.ipynb").read_text(encoding="utf-8"))
_ref = [l for l in "".join("".join(o.get("text", "")) for o in _ref["cells"][4].get("outputs", [])).splitlines()
        if l.startswith("r") and "| test" in l]
_mine = [f"r{s['repeat']} f{s['fold']}: train {len(s['train']):5d}w/{_ns(s['train']):2d}s ({_cls(s['train'])}) | "
         f"test {len(s['test']):4d}w/{_ns(s['test']):2d}s ({_cls(s['test'])})   [HC/AD/FTD]" for s in SPLITS]
_n = min(len(_ref), len(_mine))
assert _n and _mine[:_n] == _ref[:_n], "folds differ from GAT_multiclass_ds004504.ipynb"
print(f"folds identical to GAT_multiclass_ds004504.ipynb ({_n // N_FOLDS} repeats checked)")
print(f"each repeat tests all {len(Y):,} windows / {N_SUBJECTS} subjects exactly once")

r1 f1: train  6390w/70s (23/29/18) | test 1660w/18s (6/7/5)   [HC/AD/FTD]
r1 f2: train  6398w/70s (23/29/18) | test 1652w/18s (6/7/5)   [HC/AD/FTD]
r1 f3: train  6432w/70s (23/28/19) | test 1618w/18s (6/8/4)   [HC/AD/FTD]
r1 f4: train  6490w/71s (24/29/18) | test 1560w/17s (5/7/5)   [HC/AD/FTD]
r1 f5: train  6490w/71s (23/29/19) | test 1560w/17s (6/7/4)   [HC/AD/FTD]
r2 f1: train  6390w/70s (23/29/18) | test 1660w/18s (6/7/5)   [HC/AD/FTD]
r2 f2: train  6398w/70s (23/29/18) | test 1652w/18s (6/7/5)   [HC/AD/FTD]
r2 f3: train  6432w/70s (23/28/19) | test 1618w/18s (6/8/4)   [HC/AD/FTD]
r2 f4: train  6490w/71s (24/29/18) | test 1560w/17s (5/7/5)   [HC/AD/FTD]
r2 f5: train  6490w/71s (23/29/19) | test 1560w/17s (6/7/4)   [HC/AD/FTD]
r3 f1: train  6398w/70s (23/29/18) | test 1652w/18s (6/7/5)   [HC/AD/FTD]
r3 f2: train  6432w/70s (23/28/19) | test 1618w/18s (6/8/4)   [HC/AD/FTD]
r3 f3: train  6490w/71s (23/29/19) | test 1560w/17s (6/7/4)   [HC/AD/FTD]
r3 f4: train  6390w/70s (23/29/18) | t

In [5]:
# ===========================================================================
# CELL 5 — THE BAR: band power -> multinomial logistic regression, same subjects
# ===========================================================================
# 95 numbers per window (5 bands x 19 electrodes of mean log power), no graph.
# Fitted on exactly the training subjects the network uses, tested on the same
# test subjects, scored the same way.
feat = BP.reshape(len(BP), -1)
FLOOR_OOF = np.full((N_REPEATS, len(Y), N_CLASSES), np.nan)
for sp in SPLITS:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000, class_weight="balanced"))
    FLOOR_OOF[sp["repeat"] - 1, sp["test"]] = (clf.fit(feat[sp["train"]], Y[sp["train"]])
                                               .predict_proba(feat[sp["test"]]))

FLOOR, FLOOR_SD = mean_sd(pd.DataFrame([score(np.arange(len(Y)), FLOOR_OOF[r]) for r in range(N_REPEATS)]))
print(f"linear floor, subject level : accuracy {FLOOR['s_acc']:.4f}  sensitivity {FLOOR['s_sensitivity']:.3f}  "
      f"specificity {FLOOR['s_specificity']:.3f}  | AUC {FLOOR['s_auc']:.4f}")

linear floor, subject level : accuracy 0.8068  sensitivity 0.825  specificity 0.770  | AUC 0.8782


---
## Model

The code is copied unchanged from `GAT_ds004504.ipynb`. The only edit is the
last linear layer, which now outputs `N_CLASSES = 2` logits. See that notebook
for why the attention carries a `+ log E` connectivity prior.


In [6]:
# ===========================================================================
# CELL 6 — MODEL
# ===========================================================================
class BandCNN(nn.Module):
    """Paper Eq. 6: one Conv1d(1, 1, D_f) per band, shared across the 19 electrodes.

    D_f = F_f - 16 + 1, so every band comes out 16 wide and keeps its frequency order.
    """
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv1d(1, 1, BAND_DIMS[b] - CNN_OUT_DIM + 1)
                                    for b in BAND_NAMES])

    def forward(self, x, band):                    # (B, 19, F_f) -> (B, 19, 16)
        b, n, f = x.shape
        return self.convs[band](x.reshape(b * n, 1, f)).reshape(b, n, CNN_OUT_DIM)


class GATLayer(nn.Module):
    """Classic multi-head GAT (Velickovic et al. 2018), dense over 19 electrodes.

        h_i   = W x_i                                    (per head)
        e_ij  = LeakyReLU(a_src . h_i + a_dst . h_j) + log E_ij
        alpha = softmax_j(e_ij)
        out_i = mean over heads of  sum_j alpha_ij h_j
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.heads, self.out_dim = HEADS, out_dim
        self.W     = nn.Linear(in_dim, HEADS * out_dim, bias=False)
        self.a_src = nn.Parameter(torch.empty(HEADS, 1, out_dim))
        self.a_dst = nn.Parameter(torch.empty(HEADS, 1, out_dim))
        self.bias  = nn.Parameter(torch.zeros(out_dim))
        self.drop  = nn.Dropout(ATTN_DROPOUT)
        for w in (self.W.weight, self.a_src, self.a_dst):
            nn.init.xavier_uniform_(w, gain=math.sqrt(2.0))

    def forward(self, x, adj):                     # x (B, 19, in), adj (B, 19, 19)
        b, n, _ = x.shape
        h   = self.W(x).view(b, n, self.heads, self.out_dim).transpose(1, 2)  # (B, H, 19, d)
        src = (h * self.a_src).sum(-1)                                        # (B, H, 19)
        dst = (h * self.a_dst).sum(-1)
        e   = F.leaky_relu(src.unsqueeze(-1) + dst.unsqueeze(-2), 0.2)        # (B, H, 19, 19)
        e   = e + torch.log(adj.clamp(min=1e-8)).unsqueeze(1)   # clamp: log(0) = -inf breaks softmax
        alpha = self.drop(torch.softmax(e, dim=-1))
        return (alpha @ h).mean(1) + self.bias                                # (B, 19, d)


class BandGAT(nn.Module):
    """Eq. 8 with attention in place of convolution: Pearson graph, then MI graph."""
    def __init__(self):
        super().__init__()
        self.g1 = GATLayer(CNN_OUT_DIM, GAT_HIDDEN)
        self.g2 = GATLayer(GAT_HIDDEN, GAT_OUT)

    def forward(self, x, e1, e2):                  # (B, 19, 16) -> (B, 19, 5)
        return F.elu(self.g2(F.elu(self.g1(x, e1)), e2))


class MFEFCGAT(nn.Module):
    """BandCNN -> per-band BandGAT -> concat 5 x 19 x 5 = 475 (Eq. 9) -> MLP."""
    def __init__(self):
        super().__init__()
        self.cnn  = BandCNN()
        self.gats = nn.ModuleList([BandGAT() for _ in BAND_NAMES])
        self.head = nn.Sequential(
            nn.Linear(len(BAND_NAMES) * N_NODES * GAT_OUT, MLP_HIDDEN[0]), nn.ReLU(),
            nn.Dropout(MLP_DROPOUT[0]),
            nn.Linear(MLP_HIDDEN[0], MLP_HIDDEN[1]), nn.ReLU(),
            nn.Dropout(MLP_DROPOUT[1]),
            nn.Linear(MLP_HIDDEN[1], N_CLASSES))

    def forward(self, xs, e1, e2):                 # xs: 5 x (B, 19, F_f); e1, e2: (B, 5, 19, 19)
        z = [self.gats[i](self.cnn(xs[i], i), e1[:, i], e2[:, i]).flatten(1)
             for i in range(len(BAND_NAMES))]
        return self.head(torch.cat(z, dim=1))      # logits (B, N_CLASSES)


_m  = MFEFCGAT()
_xs = [torch.randn(8, N_NODES, BAND_DIMS[b]) for b in BAND_NAMES]
_e  = torch.rand(8, len(BAND_NAMES), N_NODES, N_NODES)
print(f"parameters : {sum(p.numel() for p in _m.parameters()):,}   (binary GAT: 22,566)")
print(f"smoke test : logits {tuple(_m(_xs, _e, _e).shape)}")
del _m, _xs, _e

parameters : 22,566   (binary GAT: 22,566)
smoke test : logits (8, 2)


In [7]:
# ===========================================================================
# CELL 7 — TRAINING
# ===========================================================================
def batches(idx, size=BATCH_SIZE, shuffle=False, rng=None):
    order = rng.permutation(len(idx)) if shuffle else np.arange(len(idx))
    for s in range(0, len(idx), size):
        sel = torch.as_tensor(idx[order[s:s + size]], device=DEVICE)
        yield [x[sel] for x in Xt], E1t[sel], E2t[sel], Yt[sel]


@torch.no_grad()
def predict(model, idx):
    """Class probabilities (n, 3) for each window, in the order of idx."""
    model.eval()
    return torch.cat([torch.softmax(model(xs, e1, e2), 1)
                      for xs, e1, e2, _ in batches(idx, size=512)]).cpu().numpy()


def class_weights(idx):
    """n / (K * n_class) on the training windows; all ones if disabled."""
    if not CLASS_WEIGHTED_LOSS:
        return None
    counts = np.bincount(Y[idx], minlength=N_CLASSES)
    return torch.tensor(len(idx) / (N_CLASSES * counts), dtype=torch.float32, device=DEVICE)


def train_fold(split):
    """Train on every non-test subject for exactly EPOCHS epochs, then predict the test subjects.

    No validation set and no early stopping: the budget is fixed in advance, so
    all ~70 non-test subjects are used for training, as they are for the floor.
    """
    seed = SEED + 1000 * split["repeat"] + split["fold"]
    seed_all(seed)
    rng = np.random.default_rng(seed)

    model = MFEFCGAT().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 0.01)
    w     = class_weights(split["train"])

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        lr, total = opt.param_groups[0]["lr"], 0.0
        for xs, e1, e2, y in batches(split["train"], shuffle=True, rng=rng):
            loss = F.cross_entropy(model(xs, e1, e2), y, weight=w)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            total += loss.item() * len(y)
        sched.step()
        history.append(dict(epoch=epoch, lr=lr, train_loss=total / len(split["train"])))

    return dict(test_prob=predict(model, split["test"]), history=pd.DataFrame(history))


print("training functions ready")

training functions ready


In [8]:
# ===========================================================================
# CELL 8 — CROSS-VALIDATION
# ===========================================================================
rows, HISTORY = [], {}
OOF = np.full((N_REPEATS, len(Y), N_CLASSES), np.nan)   # out-of-fold [P(HC), P(dementia)]
t0 = time.time()

for sp in SPLITS:
    ts = time.time()
    r = train_fold(sp)
    OOF[sp["repeat"] - 1, sp["test"]] = r["test_prob"]
    m = score(sp["test"], r["test_prob"])
    rows.append(dict(repeat=sp["repeat"], fold=sp["fold"], seconds=round(time.time() - ts), **m))
    HISTORY[(sp["repeat"], sp["fold"])] = r["history"]
    r["history"].to_csv(RESULT_DIR / f"history_r{sp['repeat']}_f{sp['fold']}.csv", index=False)
    print(f"r{sp['repeat']} f{sp['fold']}  subject acc {m['s_acc']:.4f} (HC {m['s_specificity']:.2f} "
          f"dementia {m['s_sensitivity']:.2f})  AUC {m['s_auc']:.4f}  "
          f"final loss {r['history'].train_loss.iloc[-1]:.3f}  [{rows[-1]['seconds']}s]", flush=True)

FOLDS = pd.DataFrame(rows)
FOLDS.to_csv(RESULT_DIR / "fold_results.csv", index=False)
np.save(RESULT_DIR / "oof_prob.npy", OOF)

# Pooled out-of-fold: every repeat scores all 88 subjects once. Quote these, as mean +- SD over repeats.
POOLED = pd.DataFrame([score(np.arange(len(Y)), OOF[r]) for r in range(N_REPEATS)])
POOLED.insert(0, "repeat", range(1, N_REPEATS + 1))
POOLED.to_csv(RESULT_DIR / "pooled_by_repeat.csv", index=False)
P, P_SD = mean_sd(POOLED.drop(columns="repeat"))
pm = lambda k: f"{P[k]:.4f}" + (f" +- {P_SD[k]:.4f}" if N_REPEATS > 1 else "")

print(f"\nPOOLED  subject accuracy {pm('s_acc')}  balanced {pm('s_bal_acc')}  AUC {pm('s_auc')}")
print(f"POOLED  sensitivity (dementia found) {pm('s_sensitivity')}  specificity (HC found) {pm('s_specificity')}")
print(f"POOLED  AUC within groups: AD-HC {pm('s_auc_AD_HC')}  FTD-HC {pm('s_auc_FTD_HC')}")
print(f"POOLED  window  accuracy {pm('w_acc')}  AUC {pm('w_auc')}   [{(time.time() - t0) / 60:.1f} min]")

_ = (RESULT_DIR / "summary.json").write_text(json.dumps(dict(
    model="MFE-FCGAT dementia vs HC", classes=CLASS_NAMES,
    protocol=f"{N_REPEATS} x {N_FOLDS}-fold grouped by subject (the 3-class notebook's folds); fixed {EPOCHS} "
             "epochs on all non-test subjects (no validation set); floor fitted on the same subjects",
    pooled=P.to_dict(), pooled_sd=P_SD.to_dict(), floor=FLOOR.to_dict(), floor_sd=FLOOR_SD.to_dict(),
    config=dict(LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS,
                GRAD_CLIP=GRAD_CLIP, CLASS_WEIGHTED_LOSS=CLASS_WEIGHTED_LOSS, CNN_OUT_DIM=CNN_OUT_DIM,
                GAT_HIDDEN=GAT_HIDDEN, GAT_OUT=GAT_OUT, HEADS=HEADS, ATTN_DROPOUT=ATTN_DROPOUT,
                MLP_HIDDEN=MLP_HIDDEN, MLP_DROPOUT=MLP_DROPOUT,
                N_FOLDS=N_FOLDS, N_REPEATS=N_REPEATS, SEED=SEED)), indent=2))

r1 f1  subject acc 0.7778 (HC 0.83 dementia 0.75)  AUC 0.8750  final loss 0.243  [108s]


r1 f2  subject acc 0.8333 (HC 0.83 dementia 0.83)  AUC 0.9583  final loss 0.260  [108s]


r1 f3  subject acc 0.8889 (HC 0.67 dementia 1.00)  AUC 0.9583  final loss 0.293  [106s]


r1 f4  subject acc 0.8824 (HC 0.80 dementia 0.92)  AUC 0.9833  final loss 0.289  [107s]


r1 f5  subject acc 0.7059 (HC 0.67 dementia 0.73)  AUC 0.7879  final loss 0.243  [107s]


r2 f1  subject acc 0.9444 (HC 0.83 dementia 1.00)  AUC 0.9583  final loss 0.290  [106s]


... (6 outputs left out)

r3 f3  subject acc 1.0000 (HC 1.00 dementia 1.00)  AUC 1.0000  final loss 0.327  [105s]


r3 f4  subject acc 0.7222 (HC 0.50 dementia 0.83)  AUC 0.8194  final loss 0.235  [104s]


r3 f5  subject acc 0.6471 (HC 0.60 dementia 0.67)  AUC 0.7500  final loss 0.203  [106s]



POOLED  subject accuracy 0.8144 +- 0.0174  balanced 0.8002 +- 0.0129  AUC 0.8868 +- 0.0139
POOLED  sensitivity (dementia found) 0.8418 +- 0.0259  specificity (HC found) 0.7586 +- 0.0000
POOLED  AUC within groups: AD-HC 0.9227 +- 0.0249  FTD-HC 0.8306 +- 0.0075
POOLED  window  accuracy 0.7478 +- 0.0017  AUC 0.7980 +- 0.0112   [33.6 min]


In [9]:
# ===========================================================================
# CELL 9 — RESULTS
# ===========================================================================
COLS  = ["s_acc", "s_bal_acc", "s_sensitivity", "s_specificity", "s_auc", "s_auc_AD_HC", "s_auc_FTD_HC"]
HEADS = ["acc", "bal acc", "sens", "spec", "AUC", "AUC AD-HC", "AUC FTD-HC"]


def table(title, entries):
    print("=" * 114)
    print(f"{title:37s}" + "".join(f"{h:>11s}" for h in HEADS))
    print("-" * 114)
    for name, row, sd in entries:
        print(f"{name:37s}" + "".join(f"{row[c]:11.4f}" for c in COLS))
        if N_REPEATS > 1:
            print(f"{'   +- SD over repeats':37s}" + "".join(f"{sd[c]:11.4f}" for c in COLS))
    print("=" * 114)


table("subject level, pooled over 88", [(f"MFE-FCGAT dementia (lr {LR:g})", P, P_SD),
                                        ("band-power logistic regression", FLOOR, FLOOR_SD)])
print("sens = dementia (AD or FTD) called dementia; spec = HC called HC. Benchmark: 80.7 % accuracy (Stefanou 2025).")

# --- subject-level confusion matrix and per-subject calls, mean over repeats ----
subs, s_y3, _ = subject_probs(np.arange(len(Y)), OOF[0])
S_P    = np.stack([subject_probs(np.arange(len(Y)), OOF[r])[2] for r in range(N_REPEATS)])   # (R, 88, 2)
s_pred = S_P.argmax(2)
s_y    = (s_y3 != HC).astype(int)
CM = pd.DataFrame(np.mean([confusion_matrix(s_y, s_pred[r], labels=CLASSES) for r in range(N_REPEATS)], axis=0),
                  index=[f"true {n}" for n in CLASS_NAMES], columns=[f"called {n}" for n in CLASS_NAMES])
print(f"\nsubject-level confusion matrix, mean over {N_REPEATS} repeats\n{CM.round(1)}")
s_P = S_P.mean(0)
PER_SUBJECT = pd.DataFrame(dict(subject=subs, group=np.array(["HC", "AD", "FTD"])[s_y3],
                                called=np.array(CLASS_NAMES)[s_P.argmax(1)], p_dementia=s_P[:, 1],
                                correct_runs=(s_pred == s_y).sum(0)))
PER_SUBJECT.to_csv(RESULT_DIR / "per_subject.csv", index=False)
print(f"\nmissed most often (correct in fewer than half the repeats):")
print(PER_SUBJECT[PER_SUBJECT.correct_runs < N_REPEATS / 2].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

subject level, pooled over 88                acc    bal acc       sens       spec        AUC  AUC AD-HC AUC FTD-HC
------------------------------------------------------------------------------------------------------------------
MFE-FCGAT dementia (lr 0.003)             0.8144     0.8002     0.8418     0.7586     0.8868     0.9227     0.8306
   +- SD over repeats                     0.0174     0.0129     0.0259     0.0000     0.0139     0.0249     0.0075
band-power logistic regression            0.8068     0.7975     0.8249     0.7701     0.8782     0.9010     0.8426
   +- SD over repeats                     0.0114     0.0048     0.0259     0.0199     0.0144     0.0119     0.0234
sens = dementia (AD or FTD) called dementia; spec = HC called HC. Benchmark: 80.7 % accuracy (Stefanou 2025).



subject-level confusion matrix, mean over 3 repeats
               called HC  called dementia
true HC             22.0              7.0
true dementia        9.3             49.7

missed most often (correct in fewer than half the repeats):
subject group   called  p_dementia  correct_runs
sub-006    AD       HC       0.499             1
sub-010    AD       HC       0.477             1
sub-025    AD       HC       0.377             1
sub-036    AD       HC       0.077             0
sub-037    HC dementia       0.794             0
sub-043    HC dementia       0.789             0
sub-045    HC dementia       0.771             0
sub-059    HC dementia       0.796             0
sub-061    HC dementia       0.543             1
sub-062    HC dementia       0.547             0
sub-070   FTD       HC       0.427             0
sub-071   FTD       HC       0.449             1
sub-073   FTD dementia       0.500             1
sub-080   FTD       HC       0.008             0
sub-082   FTD       HC   

---
## Ensemble: network + band power (soft vote)

The graph network and the band-power logistic regression make different
mistakes, so averaging their probabilities helps. The weights are equal and
fixed in advance, so nothing is tuned on test subjects. Both models are scored
on the same test subjects. The cell reads the network's saved out-of-fold
probabilities, so after a restart it only needs the cells above it, not
another training run.

In [10]:
# ===========================================================================
# CELL 10 — ENSEMBLE: GAT + band-power floor (soft vote, no extra training)
# ===========================================================================
NET_OOF = np.load(RESULT_DIR / "oof_prob.npy")
assert NET_OOF.shape == FLOOR_OOF.shape, "run Cell 8 with the same N_REPEATS first"
ENS_OOF = (NET_OOF + FLOOR_OOF) / 2
np.save(RESULT_DIR / "ensemble_oof_prob.npy", ENS_OOF)
np.save(RESULT_DIR / "floor_oof_prob.npy", FLOOR_OOF)

ENS, entries = {}, []
for name, oof in [(f"MFE-FCGAT dementia (lr {LR:g})", NET_OOF), ("band-power logistic regression", FLOOR_OOF),
                  ("soft vote: GAT + band power", ENS_OOF)]:
    m, sd = mean_sd(pd.DataFrame([score(np.arange(len(Y)), oof[r]) for r in range(N_REPEATS)]))
    ENS[name] = m
    entries.append((name, m, sd))
table("subject level, pooled over 88", entries)
_ = (RESULT_DIR / "ensemble_summary.json").write_text(json.dumps({k: v.to_dict() for k, v in ENS.items()}, indent=2))

subject level, pooled over 88                acc    bal acc       sens       spec        AUC  AUC AD-HC AUC FTD-HC
------------------------------------------------------------------------------------------------------------------
MFE-FCGAT dementia (lr 0.003)             0.8144     0.8002     0.8418     0.7586     0.8868     0.9227     0.8306
   +- SD over repeats                     0.0174     0.0129     0.0259     0.0000     0.0139     0.0249     0.0075
band-power logistic regression            0.8068     0.7975     0.8249     0.7701     0.8782     0.9010     0.8426
   +- SD over repeats                     0.0114     0.0048     0.0259     0.0199     0.0144     0.0119     0.0234
soft vote: GAT + band power               0.8258     0.8058     0.8644     0.7471     0.8977     0.9262     0.8531
   +- SD over repeats                     0.0131     0.0131     0.0169     0.0199     0.0066     0.0063     0.0083


---
## Statistics

The confidence intervals and tests, including the comparison with the 3-class model's dementia score on the same folds, are in `Final_scoreboard_ds004504.ipynb` (`docs/PREREG_2026-11-16.md`, part C).